# The Sonification Encoder — Data

**No external datasets.** Every value is a rational number produced by
`_jf(num, den, octave) = Fraction(num, den) × 440`, or an integer sample count
derived from `BEAT_SAMPLES = 22050`.

| Quantity | Produced by | Arithmetic |
|---|---|---|
| Tone frequencies | `sonification.FREQ` | exact `Fraction` |
| Just ratios | `Fraction(f, 440)` | exact |
| Collision classes | dictionary inversion | exact |
| Rest durations | `BEAT_SAMPLES // k` | integer, **floor** |
| ulp figures | `math.ulp` on this platform's float64 | float64 |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

from ValaQuenta.modules.sonification import maths as sn
from ValaQuenta.modules.sonification import SonificationModule

import math
from fractions import Fraction
from collections import defaultdict

print('engine :', 'ValaQuenta/modules/sonification')
print('python :', sys.version.split()[0])
print('A =', sn.CONCERT_A, 'Hz   sample rate =', sn.SAMPLE_RATE,
      '  beat =', sn.BEAT_SAMPLES, 'samples')

## D1 — The full tone table, as exact ratios

In [ ]:
print(f'{"name":>16} {"frequency":>12} {"ratio to A440":>14} {"float Hz":>16} {"type":>9}')
for name, f in sn.FREQ.items():
    ratio = Fraction(f, sn.CONCERT_A)
    print(f'{name:>16} {str(f):>12} {str(ratio):>14} {float(f):>16.6f} '
          f'{type(f).__name__:>9}')
print()
print('entries :', len(sn.FREQ))
print('all Fraction :', all(isinstance(v, Fraction) for v in sn.FREQ.values()))
print('any float    :', any(isinstance(v, float) for v in sn.FREQ.values()))

### D1b — Why `Fraction` and not equal temperament

Equal temperament would set `f = 440 · 2^(k/12)`, which is irrational for every
`k` not a multiple of 12. The module uses just intonation with exact rationals
instead, so every ratio is representable and every interval is exact.

In [ ]:
print(f'{"ratio":>8} {"just (exact)":>14} {"equal-tempered":>16} {"cents apart":>12}')
just = {'unison': Fraction(1,1), 'minor third': Fraction(6,5),
        'major third': Fraction(5,4), 'fourth': Fraction(4,3),
        'fifth': Fraction(3,2), 'major sixth': Fraction(5,3),
        'octave': Fraction(2,1)}
semis = {'unison':0,'minor third':3,'major third':4,'fourth':5,
         'fifth':7,'major sixth':9,'octave':12}
for label, r in just.items():
    et = 2 ** (semis[label]/12)
    cents = 1200*math.log2(float(r)/et)
    print(f'{label:>8} {str(r):>14} {et:>16.9f} {cents:>12.2f}')
print()
print('The just column is exact. The equal-tempered column is irrational')
print('except at the unison and the octave.')

## D2 — Distinctness of the code

In [ ]:
inv = defaultdict(list)
for name, f in sn.FREQ.items():
    inv[f].append(name)

print(f'named symbols      : {len(sn.FREQ)}')
print(f'distinct codes     : {len(inv)}')
print(f'injective          : {len(inv) == len(sn.FREQ)}')
print(f'colliding classes  : {sum(1 for v in inv.values() if len(v) > 1)}')
print()
print('collisions:')
for f, names in sorted(inv.items()):
    if len(names) > 1:
        print(f'  {str(f):>10} Hz  <-  {names}')

In [ ]:
# How much information survives the encoding.
import math as _m
H_in  = _m.log2(len(sn.FREQ))
H_out = _m.log2(len(inv))
print(f'log2(30 symbols)  = {H_in:.4f} bits')
print(f'log2(23 codes)    = {H_out:.4f} bits')
print(f'lost to collision = {H_in - H_out:.4f} bits per symbol')
print()
print('A decoder seeing a tone at 1320 Hz cannot tell nu_tau from charm.')

## D3 — The ω round trip

In [ ]:
def ulp_diff(got, exact):
    return 0.0 if got == exact else (got - exact)/math.ulp(exact)

worst, worst_name = 0.0, None
print(f'{"name":>16} {"f (float)":>18} {"omega":>18} {"back":>18} {"ulp":>6}')
for name, f in sn.FREQ.items():
    x = float(f)
    w = 2*math.pi*x
    back = w/(2*math.pi)
    d = ulp_diff(back, x)
    if abs(d) > worst:
        worst, worst_name = abs(d), name
    print(f'{name:>16} {x:>18.9f} {w:>18.9f} {back:>18.9f} {d:>6.1f}')
print()
print(f'worst |ulp| = {worst:.1f} at {worst_name}')

## D4 — The rest durations

`BEAT_SAMPLES = 22050`, one half-second at 44.1 kHz. Each rest is a rational
fraction of a beat, then floored to an integer number of samples.

In [ ]:
B = sn.BEAT_SAMPLES
intended = {
    'phonon':   Fraction(B, 4),
    'exciton':  Fraction(B, 2),
    'magnon':   Fraction(B*3, 8),
    'roton':    Fraction(B*3, 4),
    'plasmon':  Fraction(B),
    'gravinon': Fraction(B*sn.FIB_NUM, sn.FIB_DEN),
}

print(f'{"rest":>10} {"intended":>14} {"= samples":>13} {"stored":>8} '
      f'{"exact":>6} {"discarded":>10}')
total_lost = Fraction(0)
for name, want in intended.items():
    got = sn.QUASIPARTICLE_RESTS[name]
    lost = want - got
    total_lost += lost
    print(f'{name:>10} {str(want):>14} {float(want):>13.4f} {got:>8} '
          f'{str(want.denominator == 1):>6} {float(lost):>10.4f}')
print()
print(f'rests that are exact : '
      f'{sum(1 for w in intended.values() if w.denominator == 1)}/6')
print(f'total samples discarded per full cycle : {float(total_lost):.4f}')
print()
print(f'gravinon = {sn.FIB_NUM}/{sn.FIB_DEN} beats '
      f'= {float(Fraction(sn.FIB_NUM, sn.FIB_DEN)):.9f} beats')
print(f'  Fibonacci 144/89 -> phi = {(1+5**0.5)/2:.9f}')
print(f'  the ratio is exact as a Fraction; the sample count is not.')